# J022511-045433 · Lens modeling

Run the cells from top to bottom to execute this stage and display its scientific outputs.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/systems.json").is_file())
sys.path.insert(0, str(ROOT))
from photo_lens import available_systems, run_stage, stage_artifacts
from IPython.display import display, Image, Markdown
SYSTEM = "J022511-045433"
record = next(item for item in available_systems() if item["id"] == SYSTEM)
record


## 1. Inspect imaging and model settings
The HSC science image, variance map, PSF and positions share a common coordinate grid. The configuration defines MGE light, lens mass, source basis and pixel-source regularization.

In [ ]:
import json
config_path = ROOT / record["model_config"]
config = json.loads(config_path.read_text())
display(config)
display(Image(filename=str(ROOT / record["mugshot"])))

## 2. Evaluate the lens model
The default re-evaluates each saved maximum-likelihood tracer with AutoLens and exports exact lens-light-subtracted products. Set `REFIT = True` to perform the non-linear searches again; this can take a long time and uses the configured JAX backend.

In [ ]:
REFIT = False
artifacts = run_stage(SYSTEM, "modeling", mode="fit" if REFIT else "render")

## 3. Inspect the four AutoLens modeling subplots
The stages progress from SIE mass and MGE light to a power-law mass, two source bases, and a regularized pixelized source.

In [ ]:
for number, path in enumerate(artifacts, start=1):
    display(Markdown(f"### Stage {number}"))
    display(Image(filename=str(path)))

In [ ]:
metrics = json.loads((ROOT / "1.lens-modeling" / SYSTEM / "output/stage_previews/fit_metrics.json").read_text())
import pandas as pd
pd.DataFrame(metrics)